# IB 保证金 What-If 验证

这个 notebook 只发送 IB 的 What-If 保证金预览请求，不会发送真实订单。

使用步骤：

1. 启动 TWS 或 IB Gateway，并确认 API 端口可用。
2. 运行前两段代码，查看账户当前持仓和 `conId`。
3. 填写要模拟的合约、方向、数量和订单价格，再运行最后一段代码。
4. `initial_margin_change < 0` 代表释放初始保证金；大于 0 代表增加占用。

注意：`linear_max_quantity_estimate` 只是基于本次预览结果的线性估算。最终数量必须以新的 What-If 预览再核验一次。

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display
from ib_async import Contract
from ib_async.ib import StartupFetch

from target_treasury_monitor_clean.ib_client_lock import acquire_ib_client_lock
from target_treasury_monitor_clean.ib_session import ib_connection
from target_treasury_monitor_clean.margin_whatif import MarginWhatIfRequest, run_margin_whatif
from target_treasury_monitor_clean.settings import IBSettings, market_data_type_from_label

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

## 1. 连接参数

`CLIENT_ID` 请避免与正在运行的刷新任务重复。

In [ ]:
HOST = "127.0.0.1"
PORT = 4001
CLIENT_ID = 8321
ACCOUNT = "U16251798"
MARKET_DATA_TYPE = "delayed"

ib_settings = IBSettings(
    host=HOST,
    port=PORT,
    client_id=CLIENT_ID,
    account=ACCOUNT,
    market_data_type=market_data_type_from_label(MARKET_DATA_TYPE),
)

## 2. 查看当前持仓

从这里复制希望模拟的合约 `conId`。若平仓，请按真实下单方向填写：平多仓用 `SELL`，平空仓用 `BUY`。

In [ ]:
with acquire_ib_client_lock(HOST, PORT, CLIENT_ID, purpose="margin What-If notebook positions"):
    with ib_connection(
        ib_settings,
        fetch_fields=StartupFetch.POSITIONS | StartupFetch.ACCOUNT_UPDATES,
    ) as ib:
        rows = []
        for position in ib.positions():
            if position.account != ACCOUNT or not position.position:
                continue
            contract = position.contract
            rows.append({
                "conId": int(contract.conId or 0),
                "symbol": contract.symbol,
                "localSymbol": contract.localSymbol,
                "secType": contract.secType,
                "expiry": contract.lastTradeDateOrContractMonth,
                "strike": contract.strike,
                "right": contract.right,
                "exchange": contract.exchange,
                "position": float(position.position),
            })

positions_frame = pd.DataFrame(rows).sort_values(
    ["symbol", "secType", "expiry", "strike", "right"],
    ignore_index=True,
) if rows else pd.DataFrame()
display(positions_frame)

## 3. 填写模拟订单

新开仓和当前持仓互相抵扣时，IB 会按整个账户计算。相反，平掉对冲腿后保证金可能增加，这也是有效结果。

In [ ]:
CON_ID = 0               # 从上方持仓表复制；必须改为正整数
EXCHANGE = "CBOT"      # 例如 CBOT；非 CBOT 合约请按 IB 合约填写
ACTION = "SELL"         # BUY 或 SELL
QUANTITY = 1
ORDER_TYPE = "MKT"      # MKT 或 LMT
LIMIT_PRICE = None       # LMT 必填，例如 0.03125

if int(CON_ID) <= 0:
    raise ValueError("请先将 CON_ID 改成上方持仓表中的正整数 conId")

## 4. 点击运行本格进行保证金预览

本格会建立一次短连接，调用 IB What-If 后立即断开；不会创建或传输真实订单。

In [ ]:
request = MarginWhatIfRequest(
    contract=Contract(conId=int(CON_ID), exchange=EXCHANGE),
    action=ACTION,
    quantity=float(QUANTITY),
    order_type=ORDER_TYPE,
    limit_price=LIMIT_PRICE,
)

with acquire_ib_client_lock(HOST, PORT, CLIENT_ID, purpose="margin What-If notebook preview"):
    with ib_connection(ib_settings, fetch_fields=StartupFetch.ACCOUNT_UPDATES) as ib:
        result = run_margin_whatif(ib, ACCOUNT, request)

result_data = result.to_dict()
summary_rows = [
    ("初始保证金变化", result_data["initial_margin_change"]),
    ("初始保证金释放", result_data["initial_margin_released"]),
    ("维持保证金变化", result_data["maintenance_margin_change"]),
    ("维持保证金释放", result_data["maintenance_margin_released"]),
    ("预估可用资金变化", result_data["estimated_available_funds_change"]),
    ("预估成交后可用资金", result_data["estimated_available_funds_after"]),
    ("预估流动性缓冲变化", result_data["estimated_excess_liquidity_change"]),
    ("线性估算最多可开张数", result_data["linear_max_quantity_estimate"]),
]
display(pd.DataFrame(summary_rows, columns=["指标", f"数值 ({result.currency or '账户基准币种'})"]))

if result.warning_text:
    print("IB 风险提示：", result.warning_text)
print(result_data["linear_max_quantity_note"])